# Neighborhood-based Collaborative Filtering Recommender Systems
**Prepared by Christian Alis**

In [1]:
import numpy as np
import pandas as pd
from numpy.testing import (assert_equal, assert_almost_equal,
                           assert_array_equal, assert_array_almost_equal)

Recommender (or recommendation) systems is one of the more popular applications of Big Data and was popularized by Netflix, Amazon and Facebook. Two entities are usually handled by recommender systems: users and items. Users prefer items differently and their degree of preference is expressed by their rating also known as utility for each item. Most of the time, however, the ratings are unknown and it is up to the recommender system to predict the ratings. The data on preferences are usually displayed as a user-item matrix known as a utility matrix. To illustrate, consider the utility matrix below.

| User |   HP1  |   HP2  |   HP3  |   TW   |   SW1  |   SW2  |  SW3   |
|------|--------|--------|--------|--------|--------|--------|--------|
|  A   |    4   | &nbsp; | &nbsp; |    5   |   1    | &nbsp; | &nbsp; |
|  B   |    5   |    5   |    4   | &nbsp; | &nbsp; | &nbsp; | &nbsp; |
|  C   | &nbsp; | &nbsp; | &nbsp; |    2   |   4    |   5    | &nbsp; |
|  D   |    3   | &nbsp; | &nbsp; | &nbsp; |   3    | &nbsp; | &nbsp; |

In this example, "HP" refers to Harry Potter, "TW" to "Twilight" and "SW" to Star Wars. The ratings are integers from 1 to 5, which may correspond to the number of stars a user rated each movie. Since the ratings were explicitly assigned by the users, these are known as explicit ratings.

It may happen though that the user doesn't explicitly assign a rating and we instead rely on some proxy for concluding that a user likes an item. We call this implicit rating and examples are checking whether the user watched a movie, bought an item or clicked an ad. Converting the above utility matrix to implicit rating wherein a user "likes" a movie if they saw it, we may get:

| User |   HP1  |   HP2  |   HP3  |   TW   |   SW1  |   SW2  |  SW3   |
|------|--------|--------|--------|--------|--------|--------|--------|
|  A   |    1   | &nbsp; | &nbsp; |    1   |   1    | &nbsp; | 1 |
|  B   |    1   |    1   |    1   | &nbsp; | &nbsp; | &nbsp; | &nbsp; |
|  C   | &nbsp; | 1 | &nbsp; |    1   |   1    |   1    | &nbsp; |
|  D   | 1 | &nbsp; | &nbsp; | &nbsp; | 1 | &nbsp; | 1 |

Utility matrices with implicit ratings have a positive value for items that were deemed liked by the user and empty values for those deemed not liked. Since the amount of user effort is less with implicit rating, we may get more nonempty values with it.

Regardless of whether implicit or explicit rating was used, notice that the utility matrices are mostly empty. The number of users and items are usually at least $10^5$ yet for every user, only a few would have seen or rated 10 or so items.

The task of recommender systems is to predict the value of missing items in the utility matrix. More generally, the goal is to return the top $k$ items to be recommended to users (or top $k$ users to be recommended for an item). The main business goal is to increase revenue by bringing relevant items to the attention of users. Technically, the recommended items should be:

* **Relevant**: the item should be relevant and interesting to the user
* **Novel**: the item should be something that they have not seen before
* **Serendipitous**: it should not be immediately obvious why an item is recommended hinting an element of luck
* **Diverse**: recommended items should not be too similar to each other

There are two basic types of recommender systems: content-based and collaborative filtering. In content-based recommender systems, items are selected based on their content (features) and the type of content the user likes. For example, if based on the user profile, the user likes round objects colored red, then the recommender system would go through all the item profiles and suggest a set of items that are closest to being round and red. On the other hand, collaborative filtering uses the ratings of items provided by users to suggest an item.

## Collaborative Filtering

In collaborative filtering, items are suggested based on ratings of multiple users that interact collaboratively to filter items that will be recommended to the user. It does not explicitly use the content of the items to pick a suggestion.

There are two main classes of collaborative filtering methods:

* Neighborhood-based methods also known as memory-based methods
* Model-based method e.g., latent factor models

In this notebook, we will discuss neighborhood-based collaborative filtering methods. The next notebook will discuss latent factor models.

Preferences do not typically change rapidly so recommendations are usually precomputed and missing values of the utility matrix are estimated infrequently.

### Neighborhood-based methods

The utility matrix gives us information on users, items or both. This leads us to two approaches for neighborhood-based collaborative filtering: user-based and item-based. These approaches base their recommendations on the most similar rows (users) or columns (items), respectively. 

At this point, we need to define a measure of similarity. To better understand what we expect from a measure of similarity, consider the utility matrix above which is replicated below.

In [2]:
df_utility = pd.DataFrame(
    {
        "HP1": np.array([4, 5, np.nan, np.nan]),
        "HP2": np.array([np.nan, 5, np.nan, 3]),
        "HP3": np.array([np.nan, 4, np.nan, np.nan]),
        "TW": np.array([5, np.nan, 2, np.nan]),
        "SW1": np.array([1, np.nan, 4, np.nan]),
        "SW2": np.array([np.nan, np.nan, 5, np.nan]),
        "SW3": np.array([np.nan, np.nan, np.nan, 3]),
    },
    index=list("ABCD"),
)
df_utility.replace({np.nan: ""})

,HP1,HP2,HP3,TW,SW1,SW2,SW3
A,4.0,,,5.0,1.0,,
B,5.0,5.0,4.0,,,,
C,,,,2.0,4.0,5.0,
D,,3.0,,,,,3.0


The matrix is sparse so we cannot guarantee that the conclusions that we get by just considering the known values would be the same or even similar to what we would get if we have a dense matrix. However, we obviously have no choice and we have to make do with what we have. We observe that users A and C rated two movies in common but it seems that they have opposite tastes. User A have only one commonly rated movie with B but they both gave relatively high ratings on that movie. User D seems to not have any strong preference to the two movies he/she rated. A good similarity measure should be consistent with these observations.

When measuring the similarity $\text{sim}(u,v)$ between users $u$ and $v$, we can only consider the items $I_{uv}$ that are commonly rated by $u$ and $v$. We should not impute by replacing empty ratings with some constant value. Since the resulting vector lengths would be varying, it would make sense to use cosine similarity,
$$
\text{RawCosine}(u, v) = \frac{ \sum\limits_{i \in I_{uv}}
        r_{ui} \cdot r_{vi}} {\sqrt{\sum\limits_{i
        \in I_{uv}} r_{ui}^2} \cdot \sqrt{\sum\limits_{i \in
        I_{uv}} r_{vi}^2} },
$$
where $r_{ui}$ and $r_{vi}$ is the rating of user $u$ and $v$, respectively, for item $i$.

**Problem 1**

Create a function `raw_cos` that accepts two pandas `Series` and returns the cosine similarity between the two with respect to common non-null values. Return `0` if there is no common non-null value or one of them has length `0`.

In [3]:
def raw_cos(a, b):
    """Cosine similarity between two rating vectors on their overlap.
    
        Only the indices where both `a` and `b` are non-null are used
        (``I_uv`` in the course notes); this avoids treating a missing
        rating as if it were an actual value of 0.
    
        Parameters
        ----------
        a, b : pandas.Series
            Rating vectors to compare, e.g. two rows (users) or two
            columns (items) of a utility matrix.
    
        Returns
        -------
        float
            The raw cosine similarity between `a` and `b` restricted to
            their commonly non-null entries. Returns 0 if `a` or `b` is
            empty, if they share no non-null index, or if either vector
            is all zeros on the overlap (which would otherwise divide by
            zero).
    """
    if len(a) == 0 or len(b) == 0:
        return 0

    common = a.notna() & b.notna()
    if not common.any():
        return 0

    a_common = a[common]
    b_common = b[common]
    denominator = np.sqrt((a_common ** 2).sum()) * np.sqrt((b_common ** 2).sum())
    if denominator == 0:
        return 0

    return (a_common * b_common).sum() / denominator


In [4]:
assert_almost_equal(raw_cos(df_utility.loc["A"], df_utility.loc["B"]), 1)
assert_almost_equal(
    raw_cos(df_utility.loc["A"], df_utility.loc["C"]), 0.6139406135149205
)
assert_almost_equal(raw_cos(df_utility.loc["A"], df_utility.loc["D"]), 0)

However, we see that the results using this measure don't exactly match our expectations. For example, the similarity between users A and C suggest that they have similar tastes instead of the opposite. The ratings of user B are consistently high that a high rating may not necessarily mean they like an item.

A simple workaround to these issues is to mean-center the matrix first. That is, subtract the mean rating of each user to that user's ratings. The resulting measure is the adjusted cosine similarity given by,
$$
\text{AdjCosine}(u, v) = \frac{ \sum\limits_{i \in I_{uv}}
        (r_{ui} -  \mu_u) \cdot (r_{vi} - \mu_{v})} {\sqrt{\sum\limits_{i
        \in I_{uv}} (r_{ui} -  \mu_u)^2} \cdot \sqrt{\sum\limits_{i \in
        I_{uv}} (r_{vi} -  \mu_{v})^2} },
$$
where $\mu_u$ and $\mu_v$ correspond to the mean rating of user $u$ and $v$, respectively. Note that this is the same as taking the Pearson correlation between the ratings of $u$ and $v$, considering only commonly rated items,
$$
\text{Pearson}(u, v) = \frac{ \sum\limits_{i \in I_{uv}}
        (r_{ui} -  \mu_u) \cdot (r_{vi} - \mu_{v})} {\sqrt{\sum\limits_{i
        \in I_{uv}} (r_{ui} -  \mu_u)^2} \cdot \sqrt{\sum\limits_{i \in
        I_{uv}} (r_{vi} -  \mu_{v})^2} }.
$$

We now look at the two kinds of neighborhood-based collaborative filtering.

#### User-based Collaborative filtering

The algorithm for user-based collaborative filtering is as follows:

    for every user u:
        find k most similar users (peer group)
        for every unrated item i of u:
            set rating as the weighted average rating among the most similar users who rated that item
    
We mean-center the matrix first. To estimate the rating $\hat{r}_{ui}$ of $i$ given by $u$, we take the weighted average of the difference from the mean for the (up to) $k$ most similar users who have rated $i$ then add this to the average rating of $u$. The weight is based on the user similarity.

This can be expressed as
$$
\hat{r}_{ui} = \mu_u + \frac{ \sum\limits_{v \in N^k_i(u)}
        \text{sim}(u, v) \cdot (r_{vi} - \mu_v)} {\sum\limits_{v \in
        N^k_i(u)} \text{sim}(u, v)}.
$$
where $\mu_u$ is the mean rating of $u$, $N^k_i(u)$ is the $k$ nearest neighbors of $u$ that have rated $i$, $r_{ui}$ is the rating given by $u$ on $i$ and $\text{sim}(u, v)$ is the adjusted cosine similarity given by
$$
\text{sim}(u, v) = \frac{ \sum\limits_{i \in I_{uv}}
        (r_{ui} -  \mu_u) \cdot (r_{vi} - \mu_{v})} {\sqrt{\sum\limits_{i
        \in I_{uv}} (r_{ui} -  \mu_u)^2} \cdot \sqrt{\sum\limits_{i \in
        I_{uv}} (r_{vi} -  \mu_{v})^2} }
$$
where $I_{uv}$ is the set of items that were commonly rated by $u$ and $v$.

The $n$ recommended items would then be the $n$ unrated items that received the highest predicted ratings.

**Problem 2**

Create a function `user_complete` that accepts a utility matrix and the number $k$ of similar users to consider then returns the completed utility matrix. Preprocess the matrix by mean-centering it then use adjusted cosine similarity as similarity measure. Sort by user or item id if the distances or similarities are the same.

In [5]:
def _similarity_matrix(vectors):
    """Compute all pairwise raw-cosine similarities among a set of
    rating vectors.

    Parameters
    ----------
    vectors : pandas.DataFrame
        Rows to compare pairwise, e.g. a (mean-centered) utility
        matrix's rows for user-based similarity, or its transpose
        for item-based similarity.

    Returns
    -------
    pandas.DataFrame
        A symmetric matrix indexed and columned by ``vectors.index``,
        with entry ``(x, y)`` equal to `raw_cos` between rows `x` and
        `y` (0 on the diagonal, which is never used since no vector
        is ever its own neighbor).
    """
    labels = list(vectors.index)
    n = len(labels)
    sim = pd.DataFrame(np.zeros((n, n)), index=labels, columns=labels)
    for i in range(n):
        for j in range(i + 1, n):
            s = raw_cos(vectors.loc[labels[i]], vectors.loc[labels[j]])
            sim.iloc[i, j] = s
            sim.iloc[j, i] = s
    return sim


def _peer_group(similarity, target, k):
    """Rank every other label by similarity to `target` and keep the
    top `k`.

    Parameters
    ----------
    similarity : pandas.DataFrame
        A square similarity matrix as returned by `_similarity_matrix`.
    target : hashable
        The row/column label to find peers for.
    k : int
        The (maximum) number of peers to keep.

    Returns
    -------
    list
        Up to `k` labels other than `target`, sorted by decreasing
        similarity to `target` with ties broken by label.
    """
    others = [
        label for label in similarity.index
        if label != target and similarity.loc[target, label] > 0
    ]
    others.sort(key=lambda label: (-similarity.loc[target, label], label))
    return others[:k]


def user_complete(df_utility, k):
    """Complete a utility matrix using user-based collaborative
    filtering.

    The matrix is mean-centered by user, and adjusted cosine
    similarity (raw cosine on the centered rows) is used to find,
    for every user, a fixed peer group of the `k` most similar other
    users overall. A missing rating is then predicted from whichever
    members of that peer group happen to have rated the item, as the
    user's mean plus the similarity-weighted average deviation of
    those neighbors' ratings from their own means.

    Parameters
    ----------
    df_utility : pandas.DataFrame
        The utility matrix, users as rows and items as columns, with
        missing ratings as NaN.
    k : int
        The number of similar users in each user's peer group.

    Returns
    -------
    pandas.DataFrame
        A copy of `df_utility` with as many missing entries filled
        in as could be predicted (an entry stays NaN if nobody in
        the user's peer group rated that item). Ties in similarity
        are broken by user id.
    """
    means = df_utility.mean(axis=1, skipna=True)
    centered = df_utility.sub(means, axis=0)
    similarity = _similarity_matrix(centered)

    completed = df_utility.copy()
    for user in df_utility.index:
        peer_group = _peer_group(similarity, user, k)

        user_row = df_utility.loc[user]
        unrated_items = user_row[user_row.isna()].index

        for item in unrated_items:
            neighbors = [v for v in peer_group if pd.notna(df_utility.loc[v, item])]
            if not neighbors:
                continue

            weights = np.array([similarity.loc[user, v] for v in neighbors])
            if weights.sum() == 0:
                continue

            deviations = np.array([centered.loc[v, item] for v in neighbors])
            completed.loc[user, item] = (
                means[user] + (weights * deviations).sum() / weights.sum()
            )

    return completed


In [6]:
df_jester = pd.read_excel(
    "/mnt/data/public/jester/dataset1/jester-data-2.xls",
    header=None,
    nrows=100,
).iloc[:, 1:]
df_jester.replace(99, np.nan, inplace=True)

df_user_complete = user_complete(df_jester, 5)

assert_equal(df_user_complete.shape, (100, 100))
assert_array_almost_equal(
    df_user_complete.iloc[0, :5].tolist(),
    [-3.446504, 8.11, -1.135967, -2.830721, -2.28],
)
assert_array_almost_equal(
    df_user_complete.iloc[13, :5].tolist(),
    [np.nan, 5.8720833333333315, np.nan, np.nan, -0.49],
)

**Problem 3**

Create a function `recommend` that accepts the original and completed utility matrices then returns a list of $n$ recommended unrated items to user sorted from most recommended to least then by joke id.

In [7]:
def recommend(user, df_utility, df_completed, n):
    """Recommend the top `n` unrated items for a user.
    
        Parameters
        ----------
        user : hashable
            The label of the user (a row of `df_utility`/`df_completed`)
            to recommend items to.
        df_utility : pandas.DataFrame
            The original, incomplete utility matrix.
        df_completed : pandas.DataFrame
            A completed utility matrix aligned with `df_utility`, e.g.
            the output of `user_complete` or `item_complete`.
        n : int
            The number of items to recommend.
    
        Returns
        -------
        list
            Up to `n` item labels that `user` has not rated, ordered
            from highest to lowest predicted rating in `df_completed`,
            with ties broken by item id. Items with no prediction
            available (still NaN in `df_completed`) are excluded.
    """
    original_ratings = df_utility.loc[user]
    unrated_items = original_ratings[original_ratings.isna()].index

    predicted = df_completed.loc[user, unrated_items].dropna()
    ranked_items = sorted(predicted.index, key=lambda item: (-predicted[item], item))

    return ranked_items[:n]

In [8]:
recos = recommend(5, df_jester, df_user_complete, 10)
assert_array_equal(recos, [77, 45, 80, 11, 98, 52, 25, 85, 10, 39])

#### Item-based collaborative filtering

The algorithm for item-based collaborative filtering is similar to user-based collaborative filtering. Instead of looking for similar users first, we look for similar items then estimate the rating as the average of the user rating on those similar items:

    for every item I:
        find n most similar items
        for every user U that did not rate I:
            set rating as the weighted average rating among the most similar items rated by U
            
That is,
$$
\hat{r}_{ui} = \mu_u + \frac{ \sum\limits_{j \in N^k_u(i)}
        \text{sim}(i, j) \cdot (r_{uj} - \mu_u)} {\sum\limits_{j \in
        N^k_u(i)} \text{sim}(i, j)}
$$
where $N^k_u(i)$ is the $k$ nearest neighbors of $i$ that been rated by $u$.

**Problem 4**

Create a function `item_complete` that accepts a utility matrix and the number $k$ of similar items to consider then returns the completed utility matrix. Preprocess the matrix by mean-centering it then use adjusted cosine similarity as similarity measure. Sort by user or item id if the distances or similarities are the same.

In [9]:
def item_complete(df_utility, k):
    """Complete a utility matrix using item-based collaborative
    filtering.

    Mirrors `user_complete` on the item axis: the matrix is
    mean-centered by user, adjusted cosine similarity is computed
    between item columns of that centered matrix, and each item gets
    a fixed peer group of its `k` most similar (positively
    correlated) other items overall. A missing (user, item) rating
    is predicted from whichever peers of `item` the user actually
    rated, as the user's mean plus the similarity-weighted average
    of the user's own centered ratings on those peers.

    Parameters
    ----------
    df_utility : pandas.DataFrame
        The utility matrix, users as rows and items as columns, with
        missing ratings as NaN.
    k : int
        The number of similar items in each item's peer group.

    Returns
    -------
    pandas.DataFrame
        A copy of `df_utility` with as many missing entries filled
        in as could be predicted (an entry stays NaN if the user
        rated none of that item's peer group). Ties in similarity
        are broken by item id.
    """
    means = df_utility.mean(axis=1, skipna=True)
    centered = df_utility.sub(means, axis=0)
    similarity = _similarity_matrix(centered.T)

    completed = df_utility.copy()
    for item in df_utility.columns:
        peer_group = _peer_group(similarity, item, k)

        unrated_users = df_utility.index[df_utility[item].isna()]

        for user in unrated_users:
            neighbors = [j for j in peer_group if pd.notna(df_utility.loc[user, j])]
            if not neighbors:
                continue

            weights = np.array([similarity.loc[item, j] for j in neighbors])
            deviations = np.array([centered.loc[user, j] for j in neighbors])
            completed.loc[user, item] = (
                means[user] + (weights * deviations).sum() / weights.sum()
            )

    return completed

In [10]:
df_item_complete = item_complete(df_jester, 5)
assert_equal(df_item_complete.shape, (100, 100))
assert_array_almost_equal(
    df_item_complete.iloc[0,:5].tolist(), 
    [7.09, 8.11, -7.28, np.nan, -2.28])
assert_array_almost_equal(
    df_item_complete.iloc[13, :5].tolist(),
    [6.650871400844572, np.nan, 0.9247630520008987, np.nan, -0.49])

The algorithm for user-based and item-based collaborative filtering are almost the same. However, this duality of similarity is usually broken in practice  because of two things:

* We can already recommend items if we have already identified the most similar users. On the other hand, we need to take an additional step if we only have the most similar items.
* Items can be grouped into genres, for example, but users may like multiple genres. Thus, it is easier to find similar items because they belong to the same genre than find similar users who may prefer some genres in common but only individually in other genres.

## Evaluating Recommender Systems

Recommender systems can be evaluated in an online or offline manner. Online evaluation requires users to interact with the running recommender system, and their feedback is then used to evaluate the system. Users may be recruited to evaluate the system. They can also be actual users participating in an A/B test. Obviously, online evaluation takes more resources and effort to perform. It is not very useful for comparing different systems as well because access to the data is usually restricted. Offline evaluation relies on already collected data. There's a danger in it being too stale but it is the simplest to handle and, in practice, is the one used in research.

In general, recommender systems are evaluated to measure it with respect to the following:

* **Accuracy**: This measures the accuracy of the predicted ratings or rankings with respect to actual ratings or rankings and is the only aspect that we will focus on measuring in BDCC.
* **Coverage**: User-space coverage measures how many users can be provided $n$ recommendations knowing that users typically need to have a certain number of ratings before a recommendation can be provided. 
* **Confidence and Trust**: Confidence refers to how narrow the confidence interval is for a predicted rating. Trust refers to how much a user believes the predicted ratings are correct.
* **Novelty**: This measures how likely a recommender system can provide recommendations that a user is not yet aware of.
* **Serendipity**: This measures the level of surprise of successful recommendations.
* **Diversity**: The items in a recommendation list should be diverse.
* **Robustness and Stability**: The system should be able to handle attacks and can cope with changes in user behavior through time.
* **Scalability**: It should be able to perform efficiently and effectively even with large amounts of data.

Let us now look at how to measure accuracy of ratings and rankings.

### Evaluating accuracy of ratings

Predicting ratings can be thought of as a regression problem. We can therefore use measures suitable for that such as MSE, RMSE, MAE and MAPE. Cross-validation can be performed as well wherein we sample from actual ratings.

### Evaluating ranking via utility

Evaluating the performance of recommender systems based only on how accurately we can reconstruct the actual ratings is straightforward but does not completely matches the goals of a recommender systems. In practice, what usually matters is that the small set of items we actually recommend to the user is useful to them with the first items deemed to be more important.

A popular way of measuring this is via the discounted cumulative gain (DCG) and normalized discounted cumulative gain (NDCG). These are defined as

$$
DCG = \frac{1}{m} \sum_{u=1}^m \sum_{j \in I_u, v_j \leq L} \frac{2^{r_{uj}}-1}{\log_2 \left(v_j +1\right)}; \\
NDCG = \frac{DCG}{IDCG},
$$
where $L$ is the number of recommended items, $v_j \in {1...L}$ is the rank of the item in the recommendation, $r_{uj}$ is the actual rating of user $u$ on item $j$ and $I_u$ is the set of items rated by user $u$. IDCG is the idealized DCG which is DCG when the sorting follows the ground-truth rankings.

**Problem 5**

Create a function `dcg` that accepts the utility matrix and recommendation of selected users then returns the corresponding DCG.

In [11]:
def dcg(df_utility, recos):
    """Compute the (mean) discounted cumulative gain of recommendations.
    
        For each user, walks their recommendation list in rank order and
        adds a term for every recommended item the user actually rated
        (items with no known rating contribute nothing, since their true
        gain is unknown); items are discounted by the log of their rank.
    
        Parameters
        ----------
        df_utility : pandas.DataFrame
            The (ground-truth) utility matrix; row `i` must correspond to
            the user recommended to in ``recos[i]``.
        recos : list of list
            `recos[i]` is the ranked list of recommended item labels
            (best first) for the user in row `i` of `df_utility`.
    
        Returns
        -------
        float
            The DCG, averaged over the ``len(recos)`` users evaluated.
    """
    total = 0.0
    for i, user_recos in enumerate(recos):
        ratings = df_utility.iloc[i]
        for rank, item in enumerate(user_recos, start=1):
            rating = ratings.get(item, np.nan)
            if pd.notna(rating):
                total += (2 ** rating - 1) / np.log2(rank + 1)

    return total / len(recos)

In [12]:
assert_almost_equal(
    dcg(df_jester.loc[[5]], [[8, 61, 17, 42, 49, 27, 35, 7, 48, 66]]),
    23.023067518292404
)
assert_almost_equal(
    dcg(
        df_jester.loc[[5, 19]],
        [[8, 61, 17, 42, 49, 27, 35, 7, 48, 66],
         [7, 70, 32, 5, 18, 26, 53, 14, 97, 17]]
    ),
    101.86738320819285
)

**Problem 6**

Create a function `ndcg` that accepts the utility matrix and recommendation of selected users then returns the corresponding NDCG.

In [13]:
def ndcg(df_utility, recos):
    """Compute the (mean) normalized discounted cumulative gain.
    
        Normalizes `dcg` by the idealized DCG (IDCG): for each user, the
        same number of items as in their actual recommendation list, but
        taken from their own rated items sorted by true rating,
        descending (ties broken by item id).
    
        Parameters
        ----------
        df_utility : pandas.DataFrame
            The (ground-truth) utility matrix; row `i` must correspond to
            the user recommended to in ``recos[i]``.
        recos : list of list
            `recos[i]` is the ranked list of recommended item labels
            (best first) for the user in row `i` of `df_utility`.
    
        Returns
        -------
        float
            ``dcg(df_utility, recos) / dcg(df_utility, ideal_recos)``.
    """
    ideal_recos = []
    for i, user_recos in enumerate(recos):
        ratings = df_utility.iloc[i].dropna()
        ideal_order = ratings.sort_values(ascending=False).index
        ideal_recos.append(list(ideal_order[:len(user_recos)]))

    return dcg(df_utility, recos) / dcg(df_utility, ideal_recos)

In [14]:
assert_almost_equal(
    ndcg(df_jester.loc[[5]], [[8, 61, 17, 42, 49, 27, 35, 7, 48, 66]]),
    0.23169495849443336
)
assert_almost_equal(
    ndcg(
        df_jester.loc[[5, 19]],
        [[8, 61, 17, 42, 49, 27, 35, 7, 48, 66],
         [7, 70, 32, 5, 18, 26, 53, 14, 97, 17]]
    ),
    0.10609066993151361
)

# References

* J. Leskovec, A. Rajaraman and J. Ullman, "Mining of Massive Datasets 3e".
* C. Aggarwal, "Recommender Systems", 2016.